# A6: Naive RAG vs Contextual Retrieval

In [1]:
import os
import re
import json
import numpy as np
import torch
import fitz  # PyMuPDF - for reading PDF files
from transformers import AutoTokenizer, AutoModel
from groq import Groq
from dotenv import load_dotenv

d:\AIT\NLP_Assignment\NLP\A6\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the API key from .env file
load_dotenv()

True

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True


Loading the Groq API key from the `.env` file.

In [5]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready!")

Groq client ready!


## Task 1: Source Discovery & Data Preparation
### 1.1 Document Processing
Loading and extracting text from the assigned chapter PDF using PyMuPDF.

Assigned chapter: Chapter 6: Neural Networks

In [6]:
# We use PyMuPDF (fitz) to read our chapter6nlp.pdf

def load_pdf(filepath):
    """Extract all text from a PDF file page by page"""
    
    # Open the PDF file
    pdf = fitz.open(filepath)
    
    full_text = ""
    
    # Go through each page and extract text
    for page in pdf:
        full_text += page.get_text()
    
    pdf.close()
    return full_text

# Load our chapter 6 PDF
pdf_path = "data/chapter6nlp.pdf"
raw_text = load_pdf(pdf_path)

print(f"Successfully loaded PDF!")
print(f"Total characters extracted: {len(raw_text)}")
print("\n--- Preview of first 500 characters ---")
print(raw_text[:500])

Successfully loaded PDF!
Total characters extracted: 67410

--- Preview of first 500 characters ---
Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
6
Neural Networks
“[M]achines of this character can behave in a very complicated manner when
the number of units is large.”
Alan Turing (1948) “Intelligent Machines”, page 6
Neural networks are a fundamental computational tool for language process-
ing, and a very old one. They are called neural because their origins lie in the
McCulloch-Pitts neuron (McCull


Cleaning the extracted text and splitting it into smaller chunks of 500 characters with 50 character overlap for better retrieval.

In [7]:
def clean_text(text):
    """Clean the extracted PDF text"""
    
    # Replace newlines with spaces
    text = text.replace('\n', ' ')
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading and trailing spaces
    text = text.strip()
    
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    
    chunks = []
    start = 0
    
    while start < len(text):
        # Get a chunk of chunk_size characters
        end = start + chunk_size
        chunk = text[start:end]
        
        # Add the chunk to our list
        chunks.append(chunk)
        
        # Move forward by chunk_size minus overlap
        # The overlap helps preserve context between chunks
        start += chunk_size - overlap
    
    return chunks

# First clean the raw text
cleaned_text = clean_text(raw_text)
print(f"Cleaned text length: {len(cleaned_text)} characters")

# Then split into chunks
dataset = chunk_text(cleaned_text, chunk_size=500, overlap=50)
print(f"Total chunks created: {len(dataset)}")
print("\n--- Preview of first chunk ---")
print(dataset[0])
print("\n--- Preview of second chunk ---")
print(dataset[1])

Cleaned text length: 67378 characters
Total chunks created: 150

--- Preview of first chunk ---
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER 6 Neural Networks “[M]achines of this character can behave in a very complicated manner when the number of units is large.” Alan Turing (1948) “Intelligent Machines”, page 6 Neural networks are a fundamental computational tool for language process- ing, and a very old one. They are called neural because their origins lie in the McCulloch-Pitts neuron (McCull

--- Preview of second chunk ---
 origins lie in the McCulloch-Pitts neuron (McCulloch and Pitts, 1943), a simpliﬁed model of the biological neuron as a kind of computing element that could be described in terms of propositional logic. But the modern use in language processing no longer draws on these early biological inspirations. Instead, a modern neural network is a network of small computing unit

## Task 2: Technique Comparison
### 2.1 Naive RAG
Loading the embedding model `BAAI/bge-small-en-v1.5`.

In [8]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
model = AutoModel.from_pretrained("BAAI/bge-small-en-v1.5")

model = model.to(device)

print(f"Embedding model loaded!")
print(f"Model is running on: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2982.86it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded!
Model is running on: cuda:0


Building the vector database by converting each chunk into an embedding and storing them as `(chunk, embedding)` tuples.

In [9]:
VECTOR_DB = []

def get_embedding(text):
    """Convert text into a vector (embedding) using the model"""
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
   
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use CLS token embedding 
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

def add_chunk_to_database(chunk):
    """Add a chunk and its embedding to the vector database"""
    
    embedding = get_embedding(chunk)
    VECTOR_DB.append((chunk, embedding))

# Add all chunks to the database
# This is exactly like the professor's loop!
for i, chunk in enumerate(dataset):
    add_chunk_to_database(chunk)
    
    # Print progress every 30 chunks and at the last chunk
    if (i + 1) % 30 == 0 or i + 1 == len(dataset):
        print(f"Added chunk {i+1}/{len(dataset)} to the database")

print(f"\nVector database ready! Total entries: {len(VECTOR_DB)}")

Added chunk 30/150 to the database
Added chunk 60/150 to the database
Added chunk 90/150 to the database
Added chunk 120/150 to the database
Added chunk 150/150 to the database

Vector database ready! Total entries: 150


Implementing the `cosine_similarity()` and `retrieve()` functions to find the most relevant chunks for a given query.

In [10]:
def cosine_similarity(a, b):
    """Calculate how similar two embeddings are"""
    
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)


def retrieve(query, top_n=5):
    """
    Find the most relevant chunks for a given query.
    """
    
    # First convert the query into an embedding
    query_embedding = get_embedding(query)
    
    # Temporary list to store (chunk, similarity) pairs
    similarities = []
    
    for chunk, embedding in VECTOR_DB:
        similarity = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, similarity))
    
    # Sort by similarity in descending order
    # Higher similarity = more relevant chunks
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return the top N most relevant chunks
    return similarities[:top_n]


# Quick test 
test_query = "What is a neural network?"
retrieved = retrieve(test_query)

print(f"Query: {test_query}")
print(f"\nRetrieved knowledge:")
for chunk, similarity in retrieved:
    print(f" - (similarity: {similarity:.2f}) {chunk[:100]}...")

Query: What is a neural network?

Retrieved knowledge:
 - (similarity: 0.80) on learning. For that reason deep neural nets are the right tool for tasks that offer sufﬁcient data...
 - (similarity: 0.77) ue. In this chapter we introduce the neural net applied to classiﬁcation. The architecture we introd...
 - (similarity: 0.76)  origins lie in the McCulloch-Pitts neuron (McCulloch and Pitts, 1943), a simpliﬁed model of the bio...
 - (similarity: 0.76) the positive and negative cases of XOR. After Goodfellow et al. (2016). 6.3 Feedforward Neural Netwo...
 - (similarity: 0.75) s. 6.6 Training Neural Nets A feedforward neural net is an instance of supervised machine learning i...


Implementing the `generate_answer()` function that takes retrieved chunks and generates an answer using the Groq LLM (`llama-3.1-8b-instant`).

In [11]:
def generate_answer(query, retrieved_knowledge):
    """
    Generate an answer using the retrieved chunks.
    
    """ 
    instruction_prompt = f"""You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{chr(10).join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}

Question: {query}
Chatbot response:"""

    # Call Groq API instead of local Llama model
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Free model on Groq
        messages=[
            {"role": "user", "content": instruction_prompt}
        ],
        temperature=0,  # 0 = deterministic, same answer every time
        max_tokens=300
    )
    
    return response.choices[0].message.content.strip()


# Quick test
test_query = "What is a neural network?"
retrieved_knowledge = retrieve(test_query)

print("Retrieved knowledge:")
for chunk, similarity in retrieved_knowledge:
    print(f" - (similarity: {similarity:.2f}) {chunk[:100]}...")

print("\n--- Generating answer ---")
answer = generate_answer(test_query, retrieved_knowledge)
print(f"\nQuestion: {test_query}")
print(f"Answer: {answer}")

Retrieved knowledge:
 - (similarity: 0.80) on learning. For that reason deep neural nets are the right tool for tasks that offer sufﬁcient data...
 - (similarity: 0.77) ue. In this chapter we introduce the neural net applied to classiﬁcation. The architecture we introd...
 - (similarity: 0.76)  origins lie in the McCulloch-Pitts neuron (McCulloch and Pitts, 1943), a simpliﬁed model of the bio...
 - (similarity: 0.76) the positive and negative cases of XOR. After Goodfellow et al. (2016). 6.3 Feedforward Neural Netwo...
 - (similarity: 0.75) s. 6.6 Training Neural Nets A feedforward neural net is an instance of supervised machine learning i...

--- Generating answer ---

Question: What is a neural network?
Answer: A neural network is a network of small computing units, each of which takes a vector of input values and produces a single output value.


### 2.2 Contextual Retrieval
Implementing Contextual Retrieval by enriching each chunk with document-level context using the LLM before embedding.

In [13]:
# We need a separate vector database for contextual retrieval
VECTOR_DB_CONTEXTUAL = []

def enrich_chunk(chunk, document, title):
    """
    Add contextual prefix to a chunk using LLM.
    """
    
    prompt = f"""
Title: {title}
{document[:4000]}
{chunk}

Provide brief context (1-2 sentences) explaining what this chunk discusses
in relation to the full document. Format: "This chunk from [title] discusses [explanation]." """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=150
    )
    
    context = response.choices[0].message.content.strip()

    # Prepend the context to the original chunk
    return f"{context}\n\n{chunk}"


def add_chunk_to_database_contextual(chunk, document, title):
    """Add a contextually enriched chunk to the contextual vector database"""
    
    enriched_chunk = enrich_chunk(chunk, document, title)
    embedding = get_embedding(enriched_chunk)
    VECTOR_DB_CONTEXTUAL.append((enriched_chunk, embedding))


# Build the contextual vector database
print("Building contextual vector database...")


title = "Chapter 6: Neural Networks - Speech and Language Processing"

for i, chunk in enumerate(dataset):
    add_chunk_to_database_contextual(chunk, cleaned_text, title)
    
    # Print progress every 30 chunks and at the last chunk
    if (i + 1) % 30 == 0 or i + 1 == len(dataset):
        print(f"Added contextual chunk {i+1}/{len(dataset)} to the database")

print(f"\nContextual vector database ready! Total entries: {len(VECTOR_DB_CONTEXTUAL)}")

Building contextual vector database...
Added contextual chunk 30/150 to the database
Added contextual chunk 60/150 to the database
Added contextual chunk 90/150 to the database
Added contextual chunk 120/150 to the database
Added contextual chunk 150/150 to the database

Contextual vector database ready! Total entries: 150


In [14]:
import pickle

# Save the contextual vector database to a file
with open("vector_db_contextual.pkl", "wb") as f:
    pickle.dump(VECTOR_DB_CONTEXTUAL, f)

print("Vector database saved to vector_db_contextual.pkl!")

Vector database saved to vector_db_contextual.pkl!


Implementing the `retrieve_contextual()` function using the enriched vector database, and comparing it side by side with Naive RAG.

In [16]:
def retrieve_contextual(query, top_n=5):
    """
    Find the most relevant enriched chunks for a given query.

    """
    
    # First convert the query into an embedding
    query_embedding = get_embedding(query)
    
    # Temporary list to store (chunk, similarity) pairs
    similarities = []
    
    for chunk, embedding in VECTOR_DB_CONTEXTUAL:
        similarity = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, similarity))
    
    # Sort by similarity in descending order
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return the top N most relevant chunks
    return similarities[:top_n]


# Quick test to compare Naive RAG vs Contextual Retrieval
test_query = "What is a neural network?"

print("=" * 60)
print("NAIVE RAG retrieved chunks:")
print("=" * 60)
naive_retrieved = retrieve(test_query)
for chunk, similarity in naive_retrieved:
    print(f" - (similarity: {similarity:.2f}) {chunk[:100]}...")

print("\n")
print("=" * 60)
print("CONTEXTUAL RETRIEVAL retrieved chunks:")
print("=" * 60)
contextual_retrieved = retrieve_contextual(test_query)
for chunk, similarity in contextual_retrieved:
    print(f" - (similarity: {similarity:.2f}) {chunk[:100]}...")

NAIVE RAG retrieved chunks:
 - (similarity: 0.80) on learning. For that reason deep neural nets are the right tool for tasks that offer sufﬁcient data...
 - (similarity: 0.77) ue. In this chapter we introduce the neural net applied to classiﬁcation. The architecture we introd...
 - (similarity: 0.76)  origins lie in the McCulloch-Pitts neuron (McCulloch and Pitts, 1943), a simpliﬁed model of the bio...
 - (similarity: 0.76) the positive and negative cases of XOR. After Goodfellow et al. (2016). 6.3 Feedforward Neural Netwo...
 - (similarity: 0.75) s. 6.6 Training Neural Nets A feedforward neural net is an instance of supervised machine learning i...


CONTEXTUAL RETRIEVAL retrieved chunks:
 - (similarity: 0.80) This chunk from "Chapter 6: Neural Networks - Speech and Language Processing" discusses the fundamen...
 - (similarity: 0.79) This chunk from "Chapter 6: Neural Networks - Speech and Language Processing" discusses the fundamen...
 - (similarity: 0.79) This chunk from "Chapter 6:

### 1.3 QA Pair Generation
Creating 20 Question-Answer pairs based strictly on the content of Chapter 6.

In [ ]:
qa_pairs = [
    {
        "question": "What is a neural network?",
        "ground_truth_answer": "A neural network is a network of small computing units, each of which takes a vector of input values and produces a single output value."
    },
    {
        "question": "What is a bias term in a neural unit?",
        "ground_truth_answer": "A bias term is an additional term added to the weighted sum of inputs in a neural unit, represented as b in the equation z = w·x + b."
    },
    {
        "question": "What is the sigmoid activation function?",
        "ground_truth_answer": "The sigmoid function is a non-linear activation function defined as y = 1/(1+e^-z) that maps output values into the range (0,1)."
    },
    {
        "question": "What is the ReLU activation function?",
        "ground_truth_answer": "ReLU stands for Rectified Linear Unit and is defined as y = max(z,0). It is the simplest and perhaps most commonly used activation function."
    },
    {
        "question": "What is the tanh activation function?",
        "ground_truth_answer": "The tanh function is a variant of the sigmoid that ranges from -1 to +1, defined as y = (e^z - e^-z)/(e^z + e^-z)."
    },
    {
        "question": "What is the vanishing gradient problem?",
        "ground_truth_answer": "The vanishing gradient problem occurs when gradients become almost zero during backpropagation, causing the error signal to get smaller and smaller until it is too small to be used for training."
    },
    {
        "question": "Why can't a single perceptron compute XOR?",
        "ground_truth_answer": "A single perceptron cannot compute XOR because XOR is not a linearly separable function. A perceptron is a linear classifier and cannot draw a single line to separate the positive and negative cases of XOR."
    },
    {
        "question": "What is a feedforward neural network?",
        "ground_truth_answer": "A feedforward network is a multilayer network in which units are connected with no cycles. Outputs from units in each layer are passed to units in the next higher layer, and no outputs are passed back to lower layers."
    },
    {
        "question": "What does fully-connected mean in a neural network?",
        "ground_truth_answer": "Fully-connected means that each unit in each layer takes as input the outputs from all the units in the previous layer, and there is a link between every pair of units from two adjacent layers."
    },
    {
        "question": "What is the softmax function?",
        "ground_truth_answer": "The softmax function normalizes a vector of real values into a probability distribution where all numbers lie between 0 and 1 and sum to 1. It is defined as softmax(zi) = exp(zi) / sum(exp(zj))."
    },
    {
        "question": "What is an embedding matrix?",
        "ground_truth_answer": "An embedding matrix E is a dictionary of static embeddings where each row represents a token of the vocabulary as a vector of dimensionality d. It has shape [|V| x d] where |V| is the vocabulary size."
    },
    {
        "question": "What is mean pooling?",
        "ground_truth_answer": "Mean pooling is a method to combine multiple embeddings into a single embedding by summing all the embeddings and dividing by the number of tokens N."
    },
    {
        "question": "What is the cross-entropy loss function?",
        "ground_truth_answer": "The cross-entropy loss function is defined as the negative log of the output probability corresponding to the correct class, also called the negative log likelihood loss."
    },
    {
        "question": "What is error backpropagation?",
        "ground_truth_answer": "Error backpropagation is an algorithm used to compute the gradients of the loss function for a neural network by passing gradients back from the final node to all nodes in the computation graph."
    },
    {
        "question": "What is a computation graph?",
        "ground_truth_answer": "A computation graph is a representation of the process of computing a mathematical expression, where the computation is broken down into separate operations each modeled as a node in a graph."
    },
    {
        "question": "What is dropout in neural networks?",
        "ground_truth_answer": "Dropout is a regularization technique that randomly drops some units and their connections from the network during training to prevent overfitting."
    },
    {
        "question": "What are hyperparameters in neural networks?",
        "ground_truth_answer": "Hyperparameters are values chosen by the algorithm designer rather than learned by gradient descent. They include learning rate, mini-batch size, number of layers, number of hidden nodes, and choice of activation functions."
    },
    {
        "question": "What is the difference between parameters and hyperparameters?",
        "ground_truth_answer": "Parameters are the weights W and biases b that are learned by gradient descent during training. Hyperparameters are design choices like learning rate and number of layers that are tuned on a development set."
    },
    {
        "question": "What is a one-hot vector?",
        "ground_truth_answer": "A one-hot vector is a vector where all elements are 0 except one element whose dimension corresponds to the word index in the vocabulary, which has value 1."
    },
    {
        "question": "What is the chain rule in backpropagation?",
        "ground_truth_answer": "The chain rule states that the derivative of a composite function f(x) = u(v(x)) is the derivative of u with respect to v times the derivative of v with respect to x. It is used in backpropagation to compute gradients across multiple layers."
    }
]

print(f"Total QA pairs created: {len(qa_pairs)}")
print("\n--- Preview of first QA pair ---")
print(f"Q: {qa_pairs[0]['question']}")
print(f"A: {qa_pairs[0]['ground_truth_answer']}")

Total QA pairs created: 20

--- Preview of first QA pair ---
Q: What is a neural network?
A: A neural network is a network of small computing units, each of which takes a vector of input values and produces a single output value.


### 2.3 Evaluation
Running all 20 QA pairs through both Naive RAG and Contextual Retrieval pipelines to collect answers for evaluation.

In [ ]:
results = []

print("Running all 20 QA pairs through both pipelines...")
print("This will take a few minutes...\n")

for i, qa in enumerate(qa_pairs):
    
    question = qa["question"]
    ground_truth = qa["ground_truth_answer"]
    
    # ── Naive RAG ──────────────────────────────────────────
    # Step 1: Retrieve relevant chunks using normal VECTOR_DB
    naive_retrieved = retrieve(question)
    
    # Step 2: Generate answer using retrieved chunks
    naive_answer = generate_answer(question, naive_retrieved)
    
    # ── Contextual Retrieval ───────────────────────────────
    # Step 1: Retrieve relevant chunks using VECTOR_DB_CONTEXTUAL
    contextual_retrieved = retrieve_contextual(question)
    
    # Step 2: Generate answer using enriched retrieved chunks
    contextual_answer = generate_answer(question, contextual_retrieved)
    
    # ── Store result ───────────────────────────────────────
    results.append({
        "question": question,
        "ground_truth_answer": ground_truth,
        "naive_rag_answer": naive_answer,
        "contextual_retrieval_answer": contextual_answer
    })
    
    print(f"[{i+1}/20] Done: {question[:60]}...")

print("\nAll 20 QA pairs completed!")
print("\n--- Preview of first result ---")
print(f"Q: {results[0]['question']}")
print(f"Ground Truth: {results[0]['ground_truth_answer']}")
print(f"Naive RAG:    {results[0]['naive_rag_answer']}")
print(f"Contextual:   {results[0]['contextual_retrieval_answer']}")

Running all 20 QA pairs through both pipelines...
This will take a few minutes...

[1/20] Done: What is a neural network?...
[2/20] Done: What is a bias term in a neural unit?...
[3/20] Done: What is the sigmoid activation function?...
[4/20] Done: What is the ReLU activation function?...
[5/20] Done: What is the tanh activation function?...
[6/20] Done: What is the vanishing gradient problem?...
[7/20] Done: Why can't a single perceptron compute XOR?...
[8/20] Done: What is a feedforward neural network?...
[9/20] Done: What does fully-connected mean in a neural network?...
[10/20] Done: What is the softmax function?...
[11/20] Done: What is an embedding matrix?...
[12/20] Done: What is mean pooling?...
[13/20] Done: What is the cross-entropy loss function?...
[14/20] Done: What is error backpropagation?...
[15/20] Done: What is a computation graph?...
[16/20] Done: What is dropout in neural networks?...
[17/20] Done: What are hyperparameters in neural networks?...
[18/20] Done: What i

Calculating ROUGE-1, ROUGE-2, and ROUGE-L scores for both methods by comparing generated answers against the ground truth answers.

In [19]:
from rouge_score import rouge_scorer

# Initialize the ROUGE scorer
# We calculate ROUGE-1, ROUGE-2, and ROUGE-L 
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Store scores for both methods
naive_scores     = {"rouge1": [], "rouge2": [], "rougeL": []}
contextual_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for result in results:
    
    ground_truth  = result["ground_truth_answer"]
    naive_answer  = result["naive_rag_answer"]
    contextual_answer = result["contextual_retrieval_answer"]
    
    # Calculate ROUGE scores for Naive RAG
    naive_score = scorer.score(ground_truth, naive_answer)
    naive_scores["rouge1"].append(naive_score["rouge1"].fmeasure)
    naive_scores["rouge2"].append(naive_score["rouge2"].fmeasure)
    naive_scores["rougeL"].append(naive_score["rougeL"].fmeasure)
    
    # Calculate ROUGE scores for Contextual Retrieval
    contextual_score = scorer.score(ground_truth, contextual_answer)
    contextual_scores["rouge1"].append(contextual_score["rouge1"].fmeasure)
    contextual_scores["rouge2"].append(contextual_score["rouge2"].fmeasure)
    contextual_scores["rougeL"].append(contextual_score["rougeL"].fmeasure)

# Calculate average scores across all 20 questions
avg_naive = {
    "rouge1": np.mean(naive_scores["rouge1"]),
    "rouge2": np.mean(naive_scores["rouge2"]),
    "rougeL": np.mean(naive_scores["rougeL"])
}

avg_contextual = {
    "rouge1": np.mean(contextual_scores["rouge1"]),
    "rouge2": np.mean(contextual_scores["rouge2"]),
    "rougeL": np.mean(contextual_scores["rougeL"])
}


print("=" * 55)
print(f"{'Method':<25} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8}")
print("=" * 55)
print(f"{'Naive RAG':<25} {avg_naive['rouge1']:>8.4f} {avg_naive['rouge2']:>8.4f} {avg_naive['rougeL']:>8.4f}")
print(f"{'Contextual Retrieval':<25} {avg_contextual['rouge1']:>8.4f} {avg_contextual['rouge2']:>8.4f} {avg_contextual['rougeL']:>8.4f}")
print("=" * 55)

Method                     ROUGE-1  ROUGE-2  ROUGE-L
Naive RAG                   0.5653   0.3925   0.4884
Contextual Retrieval        0.5114   0.3479   0.4555


Saving all 20 QA pairs with both pipeline answers to a JSON file.

In [21]:
os.makedirs("answer", exist_ok=True)
output_filename = "answer/response-st-126686-chapter-6.json"

# Save the results list to JSON file
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"Results saved to: {output_filename}")
print(f"Total QA pairs saved: {len(results)}")

# Preview the JSON structure of first entry
print("\n--- Preview of JSON structure ---")
print(json.dumps(results[0], indent=4))

Results saved to: answer/response-st-126686-chapter-6.json
Total QA pairs saved: 20

--- Preview of JSON structure ---
{
    "question": "What is a neural network?",
    "ground_truth_answer": "A neural network is a network of small computing units, each of which takes a vector of input values and produces a single output value.",
    "naive_rag_answer": "A neural network is a network of small computing units, each of which takes a vector of input values and produces a single output value.",
    "contextual_retrieval_answer": "A neural network is a complex system composed of interconnected units or nodes, which are the building blocks of the network. These units take input values, perform computations, and produce an output. The power of neural networks comes from the ability of early layers to learn representations that can be utilized by later layers in the network."
}


## Evaluation Results & Analysis

### Models Used
- **Retriever Model:** `BAAI/bge-small-en-v1.5`
- **Generator Model:** `llama-3.1-8b-instant` via Groq API

### ROUGE Scores

| Method               | ROUGE-1 | ROUGE-2 | ROUGE-L |
|----------------------|---------|---------|---------|
| Naive RAG            | 0.5653  | 0.3925  | 0.4884  |
| Contextual Retrieval | 0.5114  | 0.3479  | 0.4555  |

### Discussion

**Naive RAG performed better** than Contextual Retrieval across all three ROUGE metrics.

**Why Naive RAG scored higher:**
- The ground truth answers are short and direct, taken straight from the textbook
- Naive RAG retrieves chunks that closely match the exact wording of the textbook
- Since ROUGE measures word overlap, direct textbook quotes score higher

**Why Contextual Retrieval scored lower:**
- Contextual Retrieval adds document-level context to each chunk
- This causes the generator to produce longer, more explanatory answers
- Longer answers with different wording score lower on ROUGE even if they are more informative

**Key Insight:**
ROUGE scores do not always reflect answer quality. Contextual Retrieval answers 
are more detailed and informative, but since they use different wording from the 
ground truth, they score lower on ROUGE. In a real-world setting, Contextual 
Retrieval would likely provide better answers to users.